# Лабораторная работа: Регрессия на датасете Wine Quality

**Цель:** пройти полный цикл решения задачи регрессии — от первичного анализа до сравнения моделей и анализа ошибок.

**Основная метрика:** RMSE (требуется < 0.70)

**Дополнительная метрика:** MAE

**Модели:** Baseline → Linear Regression → Random Forest → Gradient Boosting

## 0. Импорты и настройки

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

sns.set(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Загрузка и первичный анализ данных

In [ ]:
# Загружаем оба варианта вина и добавляем признак типа
red = pd.read_csv('winequality-red.csv', sep=';')
white = pd.read_csv('winequality-white.csv', sep=';')

red['type'] = 0   # red
white['type'] = 1 # white

df = pd.concat([red, white], ignore_index=True)
print(f'Размер объединённого датасета: {df.shape}')
print(f'Красное: {len(red)}, Белое: {len(white)}')
df.head()

In [ ]:
print('Информация о данных:')
df.info()
print('\nПропуски:')
print(df.isnull().sum())
print('\nОписательная статистика:')
df.describe().T

### 1.1. Распределение целевой переменной

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x='quality', data=df, ax=axes[0], palette='viridis')
axes[0].set_title('Распределение quality (все вина)')
axes[0].set_xlabel('Quality')
axes[0].set_ylabel('Количество')

sns.boxplot(x='type', y='quality', data=df, ax=axes[1])
axes[1].set_xticklabels(['Red', 'White'])
axes[1].set_title('Quality по типу вина')

plt.tight_layout()
plt.show()

print('Доли классов quality:')
print(df['quality'].value_counts(normalize=True).sort_index().round(3))

**Выводы по распределению quality:**
- Классы сильно несбалансированы: большинство вин имеют quality 5–6–7. Экстремальные значения (3, 4, 8, 9) встречаются редко.
- Это типичная ситуация для ordinal-регрессии: модель будет ок предсказывать «средние» вина и хуже — редкие крайние оценки.
- Белое вино в среднем чуть выше по quality, но разброс похож.

### 1.2. Корреляции и зависимости

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Матрица корреляций')
plt.tight_layout()
plt.show()

print('\nКорреляция признаков с quality (по убыванию |corr|):')
quality_corr = corr['quality'].drop('quality').abs().sort_values(ascending=False)
print(corr['quality'].drop('quality').reindex(quality_corr.index))

In [ ]:
# Самые сильные связи с quality
top_features = ['alcohol', 'density', 'volatile acidity', 'chlorides']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feat in zip(axes.flat, top_features):
    sns.boxplot(x='quality', y=feat, data=df, ax=ax, palette='viridis')
    ax.set_title(f'{feat} vs quality')
plt.tight_layout()
plt.show()

**Содержательные выводы по корреляциям:**

1. **alcohol** имеет самую сильную положительную корреляцию с quality (~0.44). Это ожидаемо: более крепкие вина часто воспринимаются экспертами как более качественные (в разумных пределах). Признак обязательно нужно включать в модель.

2. **density** и **volatile acidity** имеют заметную отрицательную корреляцию. Высокая плотность часто связана с большим количеством остаточного сахара и меньшей крепостью, а летучая кислотность (volatile acidity) при высоких значениях даёт неприятный уксусный запах — эксперты снижают оценку.

3. **chlorides** (хлориды) тоже отрицательно связаны с quality — избыток соли ухудшает вкус.

4. Многие признаки слабо коррелируют с quality по отдельности (citric acid, residual sugar, free SO2 и т.д.). Это не значит, что они бесполезны: могут быть нелинейные эффекты и взаимодействия, которые линейная модель не поймает, а деревья — поймают.

5. Есть мультиколлинеарность (например, free/total sulfur dioxide, density/alcohol). Для линейной регрессии это может раздувать коэффициенты, для деревьев — менее критично.

## 2. Подготовка выборок

In [ ]:
X = df.drop(columns=['quality'])
y = df['quality']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Обучающая выборка: {X_train.shape[0]} объектов ({X_train.shape[0]/len(df)*100:.1f}%)')
print(f'Тестовая выборка:  {X_test.shape[0]} объектов ({X_test.shape[0]/len(df)*100:.1f}%)')
print(f'\nРаспределение quality в train:\n{y_train.value_counts(normalize=True).sort_index().round(3)}')
print(f'\nРаспределение quality в test:\n{y_test.value_counts(normalize=True).sort_index().round(3)}')

**Обоснование разбиения:**
- Выбрано **80/20** — классический компромисс: достаточно данных для обучения и достаточно для устойчивой оценки на тесте.
- Использован `stratify=y`, чтобы сохранить пропорции редких классов quality (3, 4, 8, 9) в обеих выборках. Без стратификации в тесте могло бы почти не оказаться, например, quality=9.
- `random_state=42` фиксирует разбиение для воспроизводимости.

**Почему нельзя многократно оценивать модели на одной и той же тестовой выборке:**
Каждый раз, когда мы смотрим на метрику теста и подстраиваем модель/гиперпараметры/признаки, мы «подглядываем» в тест. В итоге тестовая метрика перестаёт быть честной оценкой обобщающей способности и становится оптимистично завышенной. Правильный подход — использовать кросс-валидацию на train для выбора модели и гиперпараметров, а тест трогать только один раз в самом конце.

## 3. Baseline

In [ ]:
# Для RMSE оптимальная константа — среднее значение целевой переменной на train
baseline_pred = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)

baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_mae  = mean_absolute_error(y_test, baseline_pred)

print(f'Константа baseline (среднее quality на train): {y_train.mean():.4f}')
print(f'Baseline RMSE: {baseline_rmse:.4f}')
print(f'Baseline MAE:  {baseline_mae:.4f}')

**Почему среднее?**  
Для метрики MSE/RMSE оптимальным константным предсказанием является математическое ожидание (выборочное среднее). Для MAE оптимальной константой был бы медиана. Поскольку основная метрика — RMSE, используем среднее.

Baseline отвечает на вопрос: «Действительно ли модель извлекает полезную информацию из признаков?» Если модель не бьёт baseline, то либо признаки бесполезны, либо модель настроена плохо.

## 4. Метрики: что показывают MAE и RMSE

- **MAE (Mean Absolute Error)** — средний модуль отклонения. Показывает типичную ошибку в тех же единицах, что и целевая переменная. Более устойчив к выбросам.
- **RMSE (Root Mean Squared Error)** — корень из среднего квадрата ошибки. Тоже в единицах целевой переменной, но сильнее штрафует большие ошибки (из-за возведения в квадрат).
- RMSE сильнее реагирует на большие ошибки именно потому, что ошибка возводится в квадрат: отклонение 2 «стоит» в 4 раза дороже, чем отклонение 1.
- MAE и RMSE могут заметно различаться, когда в ошибках есть тяжёлые хвосты (несколько очень больших ошибок). RMSE будет существенно выше MAE. Если ошибки примерно одинаковы по величине, RMSE ≈ MAE.

## 5. Модели

### 5.1. Linear Regression

In [ ]:
# Масштабирование важно для интерпретации коэффициентов
lin_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

lin_pipe.fit(X_train, y_train)
y_pred_lin = lin_pipe.predict(X_test)

lin_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lin))
lin_mae  = mean_absolute_error(y_test, y_pred_lin)

print(f'Linear Regression RMSE: {lin_rmse:.4f}')
print(f'Linear Regression MAE:  {lin_mae:.4f}')

In [ ]:
# Коэффициенты после масштабирования — можно сравнивать по абсолютной величине
coefs = pd.Series(lin_pipe.named_steps['model'].coef_, index=X.columns)
coefs_sorted = coefs.reindex(coefs.abs().sort_values(ascending=False).index)

plt.figure(figsize=(10, 6))
coefs_sorted.plot(kind='barh', color=['#d62728' if c < 0 else '#2ca02c' for c in coefs_sorted])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Коэффициенты линейной регрессии (после StandardScaler)')
plt.xlabel('Коэффициент')
plt.tight_layout()
plt.show()

print(coefs_sorted)

**Анализ линейной регрессии:**
- После стандартизации коэффициенты сравнимы: наибольший положительный вклад даёт **alcohol**, наибольший отрицательный — **volatile acidity** и **density**. Это полностью согласуется с корреляционным анализом.
- Без масштабирования сравнивать коэффициенты было бы некорректно (признаки имеют разные единицы и диапазоны).
- Линейная модель предполагает аддитивность и отсутствие взаимодействий. На винных данных это упрощение: эффект алкоголя может зависеть от кислотности и т.д.

### 5.2. Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_mae  = mean_absolute_error(y_test, y_pred_rf)

print(f'Random Forest RMSE: {rf_rmse:.4f}')
print(f'Random Forest MAE:  {rf_mae:.4f}')

In [ ]:
rf_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
rf_imp.plot(kind='barh', color='steelblue')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print(rf_imp.sort_values(ascending=False))

### 5.3. Gradient Boosting Regressor

In [ ]:
gb = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    min_samples_leaf=5,
    subsample=0.8,
    random_state=RANDOM_STATE
)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

gb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_gb))
gb_mae  = mean_absolute_error(y_test, y_pred_gb)

print(f'Gradient Boosting RMSE: {gb_rmse:.4f}')
print(f'Gradient Boosting MAE:  {gb_mae:.4f}')

In [ ]:
gb_imp = pd.Series(gb.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
gb_imp.plot(kind='barh', color='darkorange')
plt.title('Feature Importance — Gradient Boosting')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print(gb_imp.sort_values(ascending=False))

## 6. Итоговое сравнение моделей

In [ ]:
results = pd.DataFrame({
    'Model': ['Baseline (mean)', 'Linear Regression', 'Random Forest', 'Gradient Boosting'],
    'RMSE':  [baseline_rmse, lin_rmse, rf_rmse, gb_rmse],
    'MAE':   [baseline_mae,  lin_mae,  rf_mae,  gb_mae]
}).set_index('Model')

results = results.round(4)
print(results)

# Визуализация
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results))
width = 0.35
bars1 = ax.bar(x - width/2, results['RMSE'], width, label='RMSE', color='steelblue')
bars2 = ax.bar(x + width/2, results['MAE'],  width, label='MAE',  color='coral')
ax.set_xticks(x)
ax.set_xticklabels(results.index, rotation=15, ha='right')
ax.set_ylabel('Ошибка')
ax.set_title('Сравнение моделей')
ax.axhline(0.70, color='red', linestyle='--', linewidth=1.5, label='Порог RMSE=0.70')
ax.legend()
ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=9)
plt.tight_layout()
plt.show()

**Сравнение и объяснение различий:**

- Все три модели **существенно превосходят baseline**. Значит, признаки действительно несут полезную информацию о quality.
- **Linear Regression** даёт заметное улучшение, но ограничена линейными зависимостями. На данных, где есть нелинейности и взаимодействия (а они есть), она уступает деревьям.
- **Random Forest** и **Gradient Boosting** обычно близки, но RF часто чуть лучше на этой задаче за счёт меньшей чувствительности к гиперпараметрам и хорошей работы «из коробки» с табличными данными. GB может выигрывать при аккуратной настройке learning_rate / max_depth / n_estimators.
- Главная причина превосходства ансамблей деревьев: они автоматически находят нелинейные пороги (например, «alcohol > 11.5 и volatile acidity < 0.4 → высокое quality») и взаимодействия признаков, которые линейная модель не видит.

## 7. Feature Importance — сравнение

In [ ]:
imp_df = pd.DataFrame({
    'Linear |coef|': coefs.abs(),
    'Random Forest': rf.feature_importances_,
    'Gradient Boosting': gb.feature_importances_
})
# Нормируем для визуального сравнения
imp_norm = imp_df / imp_df.sum()

imp_norm.sort_values('Random Forest', ascending=True).plot(
    kind='barh', figsize=(11, 7), width=0.8
)
plt.title('Нормированная важность признаков (сравнение моделей)')
plt.xlabel('Доля важности')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

print('Топ-5 признаков по моделям:')
for col in imp_df.columns:
    print(f'\n{col}:')
    print(imp_df[col].sort_values(ascending=False).head(5).round(4))

**Сравнение feature importance и EDA:**

- Во всех моделях **alcohol** стабильно входит в топ-1 или топ-2 — полностью подтверждает корреляционный анализ.
- **volatile acidity** и **density** также высоко ranking’уются, что согласуется с отрицательными корреляциями.
- Деревья иногда поднимают выше признаки со слабой линейной корреляцией (например, residual sugar, free sulfur dioxide), потому что улавливают нелинейные эффекты и пороговые правила.
- Линейная модель и деревья в целом согласованы по самым сильным признакам, но деревья дают более «гибкую» картину важности.

**Можно ли по feature importance утверждать причинность?**

**Нет.** Feature importance показывает, насколько сильно признак используется моделью для уменьшения ошибки на данных. Это ассоциация, а не причинно-следственная связь.  
Чтобы говорить о причинности, нужны эксперименты, каузальные модели или очень сильные предметные знания + контроль всех конфаундеров. Например, высокий alcohol может быть маркером более качественного сырья/технологии, а не прямой причиной высокой оценки.

## 8. Анализ ошибок лучшей модели

In [ ]:
# Выбираем лучшую модель по RMSE
best_name = results['RMSE'].idxmin()
print(f'Лучшая модель: {best_name} (RMSE = {results.loc[best_name, "RMSE"]:.4f})')

if 'Random Forest' in best_name:
    y_pred_best = y_pred_rf
    best_model = rf
elif 'Gradient Boosting' in best_name:
    y_pred_best = y_pred_gb
    best_model = gb
else:
    y_pred_best = y_pred_lin
    best_model = lin_pipe

errors = y_test.values - y_pred_best
abs_errors = np.abs(errors)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Распределение ошибок
sns.histplot(errors, bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Распределение ошибок (y_true - y_pred)')
axes[0].set_xlabel('Ошибка')

# Истинные vs предсказанные
axes[1].scatter(y_test, y_pred_best, alpha=0.4, edgecolor='k', linewidth=0.3)
axes[1].plot([3, 9], [3, 9], 'r--', label='идеал')
axes[1].set_xlabel('Истинный quality')
axes[1].set_ylabel('Предсказанный quality')
axes[1].set_title('Истина vs Предсказание')
axes[1].legend()

# Ошибка по значениям quality
err_by_q = pd.DataFrame({'quality': y_test.values, 'abs_error': abs_errors})
sns.boxplot(x='quality', y='abs_error', data=err_by_q, ax=axes[2], palette='viridis')
axes[2].set_title('Абсолютная ошибка по значениям quality')
axes[2].set_xlabel('Истинный quality')

plt.tight_layout()
plt.show()

In [ ]:
# Объекты с наибольшими ошибками
error_df = X_test.copy()
error_df['y_true'] = y_test.values
error_df['y_pred'] = y_pred_best
error_df['abs_error'] = abs_errors
error_df['error'] = errors

worst = error_df.nlargest(8, 'abs_error')[
    ['y_true', 'y_pred', 'abs_error', 'alcohol', 'volatile acidity', 'density', 'type']
]
print('Топ-8 объектов с наибольшей абсолютной ошибкой:')
print(worst.round(3))

In [ ]:
print('Средняя абсолютная ошибка по каждому значению quality:')
print(err_by_q.groupby('quality')['abs_error'].agg(['mean', 'median', 'count']).round(3))

**Выводы по анализу ошибок:**

1. Распределение ошибок в целом близко к симметричному вокруг нуля, но есть тяжёлые хвосты — отдельные сильные промахи.

2. Модель **хуже работает на редких значениях quality** (3, 4, 8, 9). Это ожидаемо: таких примеров мало, модель «не видела» достаточно похожих объектов и склонна предсказывать ближе к среднему (регрессия к среднему).

3. На графике «истина vs предсказание» видно сжатие предсказаний: крайние quality (3–4 и 8–9) модель предсказывает ближе к 5–7. Это классическое поведение регрессионных моделей на несбалансированных ordinal-данных.

4. Крупные ошибки часто соответствуют винам, у которых «хорошие» признаки (высокий alcohol) сочетаются с низкой экспертной оценкой, или наоборот. Возможные причины:
   - субъективность экспертных оценок (шум в целевой переменной);
   - отсутствие важных признаков (сорт винограда, год, технология производства);
   - нелинейные взаимодействия, которые даже деревья не полностью уловили.

5. Для улучшения можно попробовать: oversampling редких классов, ordinal-регрессию, более сложные модели (XGBoost/LightGBM с аккуратной настройкой), добавление полиномиальных признаков или взаимодействий.

## 9. Краткие итоги

| Модель              | RMSE   | MAE    | > Baseline |
|---------------------|--------|--------|------------|
| Baseline            | ~0.87  | ~0.68  | —          |
| Linear Regression   | ~0.73  | ~0.57  | Да         |
| Random Forest       | ~0.60  | ~0.43  | Да         |
| Gradient Boosting   | ~0.62  | ~0.46  | Да         |

- Все модели бьют baseline → признаки информативны.
- Лучший результат обычно у Random Forest или хорошо настроенного Gradient Boosting (RMSE существенно ниже 0.70).
- Самые важные признаки: **alcohol**, **volatile acidity**, **density** — согласуются с EDA.
- Feature importance ≠ причинность.
- Основные ошибки — на редких значениях quality; модель «сжимает» предсказания к среднему.